In this notebook we show how to pull the available feature matrix that was used for testing of the **stem.MAP** model using the ***datasets*** module from HuggingFace. We will also pull the published trained **stem.MAP model** from HuggingFace using the ***huggingfacehub*** module. The user can make predictions with their own trained model as well using the same workflow.

In [1]:
#sfisch6: AutoGluon Version 0.8.2 is needed to use with our model
from autogluon.tabular import TabularPredictor

#sfisch6: We pull the test feature matrix from huggingface using the datasets module
from datasets import load_dataset

#sfisch6: we use this module to pull our trained model from huggingface
from huggingface_hub import snapshot_download

In [4]:
#sfisch6: temporary line while the repo is still private
from getpass import getpass
from huggingface_hub import login

login(token=getpass("Hugging Face token: "))

Hugging Face token: ········


In [5]:
#sfisch6: Load train data from huggingface
dataset = load_dataset('sfisch/stem.MAP')
df_test = dataset["test"].to_pandas()

#sfisch6: first two cols are the UniProt accessions for each pair
df_test.head()

,ID,pearsonR_MES_IEX_mixedbed_20180212,pearsonR_Mallam_MES_SEC_RNAse,pearsonR_Mallam_MES_control,humap3_pearsonR_MES_IEX_mixedbed_20180212,humap3_pearsonR_Mallam_MES_SEC_RNAse,humap3_pearsonR_Mallam_MES_control,label
0,Q06138 Q0VGB7,-0.1945,-0.0743,-0.0492,-0.000958,-0.000366,-0.000242,-1
1,P99028 Q8BKF1,-0.0666,0.3617,0.3126,-0.000740,0.004021,0.003475,-1
2,Q9CZN8 Q3UV70,0.0105,0.0000,-0.0340,0.000068,0.000000,-0.000220,-1
3,Q9CZ13 Q9D2U5,0.3904,-0.0271,0.0835,0.013626,-0.000946,0.002914,-1
4,Q8BWZ3 Q9EQH4,-0.0359,0.0090,0.0927,-0.005131,0.001286,0.013248,-1


In [20]:
df_test['label'].value_counts()

-1    433046
 1      5038
Name: label, dtype: int64

In [7]:
#sfisch6: load in model
model_dir = snapshot_download(repo_id="sfisch/stem.MAP_AutoGluon")
predictor = TabularPredictor.load(f"{model_dir}/stem.MAP_AutoGluon_model_20240731")

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

/home/jupyter-sfisch6_2025/.local/lib/python3.9/site-packages/autogluon/common/utils/utils.py:74: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [10]:
#sfisch6: predict based on features - remove protein IDs
test_trim = df_test.drop(["ID"], axis=1)

#sfisch6: make sure label matches training label - best practice to make sure its the correct label
test_trim.rename(columns={'label': predictor.label}, inplace=True)

In [11]:
#sfisch6: remove test label to make predictions
y_test = test_trim[predictor.label]  # values to predict
test_nolab = test_trim.drop(columns=[predictor.label])  # delete label column 
test_nolab.head()

,pearsonR_MES_IEX_mixedbed_20180212,pearsonR_Mallam_MES_SEC_RNAse,pearsonR_Mallam_MES_control,humap3_pearsonR_MES_IEX_mixedbed_20180212,humap3_pearsonR_Mallam_MES_SEC_RNAse,humap3_pearsonR_Mallam_MES_control
0,-0.1945,-0.0743,-0.0492,-0.000958,-0.000366,-0.000242
1,-0.0666,0.3617,0.3126,-0.000740,0.004021,0.003475
2,0.0105,0.0000,-0.0340,0.000068,0.000000,-0.000220
3,0.3904,-0.0271,0.0835,0.013626,-0.000946,0.002914
4,-0.0359,0.0090,0.0927,-0.005131,0.001286,0.013248


In [14]:
#sfisch6: generate predictions
y_pred = predictor.predict(test_nolab)
print("Predictions:  \n", y_pred)

#sfisch6: evaluate predictions against true labels
perf = predictor.evaluate_predictions(y_true=y_test, y_pred=y_pred, auxiliary_metrics=True)

Predictions:  
 0        -1
1        -1
2        -1
3        -1
4        -1
         ..
438079   -1
438080   -1
438081   -1
438082   -1
438083   -1
Name: IntAct_label, Length: 438084, dtype: int64


In [15]:
#sfisch6: generate predicted probabilities
pred_test_probs = predictor.predict_proba(test_nolab)

In [16]:
#sfisch6: grab probability that interaction is true
pred_test_probs.sort_values(1)

,-1,1
222555,0.998022,0.001978
131259,0.998022,0.001978
3513,0.998022,0.001978
350301,0.998022,0.001978
249873,0.998022,0.001978
...,...,...
55790,0.012907,0.987093
160549,0.012907,0.987093
432989,0.012907,0.987093
284091,0.012907,0.987093


In [17]:
#sfisch6: annotate probabilities in original df
df_test['pred_prob'] = pred_test_probs[1]

In [18]:
df_test.sort_values("pred_prob",ascending=False)[['ID','label','pred_prob']].head(70)

,ID,label,pred_prob
166534,Q8R1B4 Q99JX4,1,0.987093
237648,P23116 Q9DCH4,1,0.987093
409095,P23116 Q9DBZ5,1,0.987093
98139,Q9QZD9 P60229,1,0.987093
275230,Q9DBZ5 Q91WK2,1,0.987093
...,...,...,...
384767,Q3UMB9 Q8C2E7,1,0.987093
139583,P60229 Q9DCH4,1,0.987093
385086,P23116 P60229,1,0.987093
213086,Q91WK2 Q8QZY1,1,0.987093


You now want to save predicted pairwise probability for downstream analysis (e.g., Precision-Recall analysis with protein_complex_maps/evaluation/plots/prcurve.py - example in [ReadMe.md]())

In [19]:
df_test.sort_values("pred_prob", ascending=False)[['ID','pred_prob']].to_csv(
    "stem.MAP_AutoGluon_test_example.pairsWprob", index=False, header=False, sep='\t')